# 2 · Build a tiny EEG vision classifier
 **Goal:** classify eyes-open versus eyes-closed resting recording conditions using real public EEG images.
 The code is supplied. Your job is to explain the representation, boundaries, curves, and mistakes.
 The condition is defined by **run number**, not by motor-imagery T1/T2 event annotations.
 Both conditions occur in every participant. This is not diagnosis or mind reading.

## Before running
Use Python 3.11 or 3.12 in Colab or local Jupyter. Code is provided: **predict → run → inspect → explain**.
Downloads and training happen only when **you** run the relevant cells. This source notebook has no precomputed results.
Use the [PhysioNet release](https://physionet.org/content/eegmmidb/1.0.0/) under its Open Data Commons Attribution License v1.0.
Cite Schalk (2009), DOI 10.13026/C28G6P, Schalk et al. (2004), DOI 10.1109/TBME.2004.827072,
and the platform citation listed on that release. Data remain outside Git.
The helper is downloaded from this course repository; inspect/review its source before running.
The default moving `main` makes starting easy; pin `SOURCE_REF` to a reviewed commit for a durable submission.
Save the exact helper bytes/hash, source notebook, package versions, configuration, and outputs.

In [ ]:
%pip -q install "numpy>=1.26,<3" "scipy>=1.11,<2" "mne>=1.7,<2" "matplotlib>=3.8,<4" "pandas>=2,<3" "Pillow>=10,<13" "scikit-learn>=1.4,<2" "ipywidgets>=8,<9"

In [ ]:
%pip -q install "tensorflow>=2.16,<3"

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, urllib.request
SOURCE_REF = "main"  # For a repeatable handoff, replace with a reviewed student-repository commit SHA.
helper = Path("chapter18_vision_studio.py")
url = ("https://raw.githubusercontent.com/ZagElSayed/Introduction-to-Neuroinformatics/"
       + SOURCE_REF + "/chapters/ch18-vision-ai/vision_studio.py")
payload = urllib.request.urlopen(url, timeout=60).read()
helper.write_bytes(payload)  # Keep this exact helper with your evidence; do not rely on a moving branch.
HELPER_HASH = hashlib.sha256(payload).hexdigest()
spec = importlib.util.spec_from_file_location("chapter18_vision_studio", helper)
studio = importlib.util.module_from_spec(spec)
spec.loader.exec_module(studio)
print("Helper SHA-256:", HELPER_HASH)
print(studio.versions())

## Lock the experiment before looking at test results
Default: subjects 1–8 train, 9–10 validate, 11–12 test; runs 1 and 2; ten nonoverlapping four-second windows per run,
starting four seconds into the recording. All derivatives follow their participant. No silent subject replacement.
The two test people yield four person-by-condition averages, **not four independent people**.
The fixed run order also leaves condition confounded with order/acquisition effects.
Do not change settings after seeing test results and keep calling that same test independent.

In [ ]:
from datetime import datetime, timezone
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = Path("outputs/ch18_studio") / RUN_TAG
SEED, EPOCH_LIMIT = 18, 15
data = studio.prepare_public_data(list(range(1, 13)), OUTPUT_DIR)
studio.write_json(OUTPUT_DIR / "helper_source.json", {"url": url, "ref": SOURCE_REF, "sha256": HELPER_HASH})
(OUTPUT_DIR / "vision_studio_source.py").write_bytes(payload)
print(data["manifest"].groupby(["participant", "condition"]).size())
print("Save this directory for Notebook 3:", OUTPUT_DIR)

In [ ]:
model = studio.make_model(SEED)
model.summary()

## Read the recipe before fitting
`64×64×1 → Conv(8) → average pool → Conv(16) → average pool → flatten → Dense(16) → one logit`.
The spatial grid is retained before flattening; frequency and time positions can matter.
The loss accepts logits; sigmoid is applied for the reported class-1 output.
Inputs are fixed pixel intensities divided by 255. Neither test data nor cohort-wide scaling fits the renderer.
**Predict:** could a two-feature numeric alpha/broad-band baseline outperform the CNN? Why?

In [ ]:
report = studio.train_studio(OUTPUT_DIR, seed=SEED, epochs=EPOCH_LIMIT)
print(json.dumps(report, indent=2))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
curves = pd.read_csv(OUTPUT_DIR / "training_curves.csv")
plt.plot(range(1,len(curves)+1), curves.loss, label="Training-batch loss", color="#167D91")
plt.plot(range(1,len(curves)+1), curves.val_loss, label="Validation loss", color="#7556A8")
plt.axvline(report["best_epoch"], color="#D87C36", ls="--", label="Selected epoch")
plt.xlabel("Training epoch"); plt.ylabel("Binary cross-entropy"); plt.legend()
plt.savefig(OUTPUT_DIR / "learning_curves.png", dpi=150, bbox_inches="tight"); plt.show()
print(pd.read_csv(OUTPUT_DIR / "test_person_condition_predictions.csv"))

## Inspect before celebrating
Compare the CNN and fixed-C logistic baseline on identical held-out participants.
The baseline uses mean log 8–12-Hz power and mean log 2–30-Hz power with **training-only** standardization.
Metrics average windows within each participant/condition and use a fixed 0.5 threshold.
Brier score assesses probability error; it does not fit a calibrator. Two test people support only a very small demonstration.
The `loss` curve records in-training batch losses, while validation is evaluated with fixed epoch-end weights.
`selected_clean_train_loss` is a separate evaluation of the selected weights.

In [ ]:
bundle_files = sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_file())
print("Evidence bundle:", bundle_files)
assert report["reload_max_probability_difference"] < 1e-5
print("Reloaded inference agrees within tolerance in this run.")
print("Notebook 3 data directory:", str(OUTPUT_DIR.resolve()))

## Explain and save
Submit the split manifest, model card, selected epoch, baseline comparison, predictions, package versions,
renderer contract, helper source/hash, and one reflection about fixed run order.
Keep EDFs, derivatives, and model binaries out of Git; follow the release's attribution requirements.
The Keras file can store final-epoch optimizer state alongside restored selected weights; full interrupted-run resumption is not established.
Use validation examples for Notebook 3's exploratory inspection; protect the test set from iterative explanation-driven tuning.
Free Colab resources vary; CPU is the default and no performance/accuracy guarantee is made.